In [1]:
import psycopg

In [2]:
%pip install psycopg[binary]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import psycopg

In [4]:
from psycopg import sql

In [5]:
%pip install random

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement random (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for random


In [6]:
import string

In [7]:
import random
import string
import pandas as pd


def generate_site_code():
    letters = "".join(random.choices(string.ascii_uppercase, k=3))
    numbers = "".join(random.choices(string.digits, k=3))
    return letters + numbers


def generate_site_data(count):
    data = []
    used_site_codes = set()
    used_coordinates = set()

    while len(data) < count:

        site_code = generate_site_code()
        latitude = round(random.uniform(-90, 90), 2)
        longitude = round(random.uniform(-180, 180), 2)

        # Skip duplicate site codes
        if site_code in used_site_codes:
            continue

        # Skip duplicate coordinates
        if (latitude, longitude) in used_coordinates:
            continue

        used_site_codes.add(site_code)
        used_coordinates.add((latitude, longitude))

        data.append(
            {
                "site_code": site_code,
                "latitude": latitude,
                "longitude": longitude,
            }
        )

    return pd.DataFrame(data)

In [8]:
def create_db_meta(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [9]:
create_db_meta("meta")

Database 'meta' created successfully!


In [10]:
def create_table_meta():
    try:
        with psycopg.connect(
            dbname="meta",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS metadata (
                        site_name VARCHAR(100) Not NULL,
                        latitude DOUBLE PRECISION Not NULL,
                        longitude DOUBLE PRECISION Not NULL
                    );
                """)

            conn.commit()
            print("metadata table created.")

    except psycopg.Error as e:
        print(e)

In [11]:
create_table_meta()

metadata table created.


In [203]:
def insert_sites():

    sites_df = generate_site_data(7000)

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:
        with conn.cursor() as cur:
            for _, row in sites_df.iterrows():
                cur.execute(
                    """
                    INSERT INTO metadata
                    (site_name, latitude, longitude)
                    VALUES (%s, %s, %s)                                                                         
                    """,
                    (row["site_code"], row["latitude"], row["longitude"]),
                )
        conn.commit()

    print("Sites inserted successfully!")

In [204]:
insert_sites()

Sites inserted successfully!


In [205]:
def get_sites():

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:

        with conn.cursor() as cur:
            cur.execute(""" 
                SELECT site_name, latitude, longitude
                FROM metadata
            """)

            sites = cur.fetchall()

    return sites

In [206]:
data = list(get_sites())

In [207]:
type(data[0])

tuple

In [170]:
# BATCH_SIZE = 30

# for i in range(0, len(data), BATCH_SIZE):

#     batch = data[i : i + BATCH_SIZE]

#     # print(len(batch))

In [171]:
# site_codes = [row[0] for row in batch]

# latitudes = [row[1] for row in batch]

# longitudes = [row[2] for row in batch]

In [44]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [209]:
import requests
from retry_requests import retry
import openmeteo_requests

session = requests.Session()

retry_session = retry(session, retries=5, backoff_factor=0.2)

openmeteo = openmeteo_requests.Client(session=retry_session)

In [ ]:
url = "https://api.open-meteo.com/v1/forecast"

import time

MAX_LOCATIONS = 7000
BATCH_SIZE = 30

locations_processed = 0

for i in range(0, len(data), 30):

    batch = data[i : i + 30]

    site_codes = [row[0] for row in batch]
    latitudes = [row[1] for row in batch]
    longitudes = [row[2] for row in batch]

    params = {
        "latitude": latitudes,
        "longitude": longitudes,
        "hourly": ["temperature_2m", "relative_humidity_2m", "direct_radiation"],
    }

    while True:
        try:
            responses = openmeteo.weather_api(url, params=params, method="POST")

            print("Batch:", i, "Responses:", len(responses))
            break  # successful request, while loop se bahar

        except Exception as e:

            if "Minutely API request limit exceeded" in str(e):

                print("API limit reached. Waiting 60 seconds...")
                time.sleep(60)

            else:
                raise e

Batch: 0 Responses: 30
Batch: 30 Responses: 30
Batch: 60 Responses: 30
Batch: 90 Responses: 30
Batch: 120 Responses: 30
Batch: 150 Responses: 30
Batch: 180 Responses: 30
Batch: 210 Responses: 30
Batch: 240 Responses: 30
Batch: 270 Responses: 30
Batch: 300 Responses: 30
Batch: 330 Responses: 30
Batch: 360 Responses: 30
Batch: 390 Responses: 30
Batch: 420 Responses: 30
Batch: 450 Responses: 30
Batch: 480 Responses: 30
Batch: 510 Responses: 30
Batch: 540 Responses: 30
Batch: 570 Responses: 30
API limit reached. Waiting 60 seconds...
Batch: 600 Responses: 30
Batch: 630 Responses: 30
Batch: 660 Responses: 30
Batch: 690 Responses: 30
Batch: 720 Responses: 30
Batch: 750 Responses: 30
Batch: 780 Responses: 30
Batch: 810 Responses: 30
Batch: 840 Responses: 30
Batch: 870 Responses: 30
Batch: 900 Responses: 30
Batch: 930 Responses: 30
Batch: 960 Responses: 30
Batch: 990 Responses: 30
Batch: 1020 Responses: 30
Batch: 1050 Responses: 30
Batch: 1080 Responses: 30
Batch: 1110 Responses: 30
Batch: 114

KeyboardInterrupt: 

In [188]:
len(responses)

500

In [189]:
for site_code, response in zip(site_codes, responses):

    hourly = response.Hourly()

    temperature = hourly.Variables(0).ValuesAsNumpy()
    humidity = hourly.Variables(1).ValuesAsNumpy()
    radiation = hourly.Variables(2).ValuesAsNumpy()

    # print(site_code, temperature[:5])

In [ ]:
df = pd.DataFrame(
        {
            "site_code": site_code,
            "time_interval": pd.date_range(
                start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=hourly.Interval()),
                inclusive="right",
            ),
            "temperature": temperature,
            "humidity": humidity,
            "solar_radiance": radiation,
        }
    )
df.to_csv("weather_data.csv", index=False)

In [191]:
def create_db_site_weather(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [192]:
create_db_site_weather("site_weather")

Database 'site_weather' created successfully!


In [193]:
def create_table_site_weather():
    try:
        with psycopg.connect(
            dbname="site_weather",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS site_weather (
                        site_name VARCHAR(100) Not NULL,
                        time_interval TIMESTAMPTZ Not NULL,
                        temperature REAL Not NULL,
                        humidity REAL Not NULL,
                        solar_radiance REAL Not NULL
                    );
                """)

            conn.commit()
            print("site_weather table created.")

    except psycopg.Error as e:
        print(e)

In [194]:
create_table_site_weather()

site_weather table created.


In [112]:
def data_store(sites_df):

    with psycopg.connect(
        dbname="site_weather",
        user="postgres",
        password="123789",
        host="localhost",
        port="5000",
    ) as conn:
        with conn.cursor() as cur:

            for _, row in sites_df.iterrows():

                cur.execute(
                    """
                    INSERT INTO site_weather
                    (site_name, time_interval, temperature, humidity, solar_radiance)
                    VALUES (%s, %s, %s, %s, %s)
                    """,
                    (
                        row["site_code"],
                        row["time_interval"],
                        row["temperature"],
                        row["humidity"],
                        row["solar_radiance"],
                    ),
                )

        conn.commit()

    print("Site weather data inserted successfully!")

In [113]:
data_store(df)

Site weather data inserted successfully!
